# 02 — Baseline 1: Plain BiGRU

First model trained in the project, on UCI-HAR's official split. This
notebook has two parts:

1. **Diagnostic single-seed run** — loss curves + confusion matrix, to sanity
   check the training loop itself before trusting any benchmark numbers.
2. **Official multi-seed benchmark run** — the actual first entries into
   `results/tables/raw_results.csv`, using the exact same code path
   (`src/train.py::train_one_run`) that every later model will go through,
   so results stay comparable.

This also exercises the params/FLOPs/latency logging added to `utils.py`
this week — every later model reuses `profile_model()` unchanged.

Prerequisite: `python -m src.preprocessing` has populated
`data/processed/ucihar/`.


## Setup

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import confusion_matrix

from src.data_loader import get_official_split, build_dataloaders
from src.train import build_model, train_one_run
from src.utils import set_seed, get_device, encode_labels, profile_model, compute_metrics, append_result_row

MODEL_NAME = "bigru_baseline"
DATASET = "ucihar"
device = get_device()
print("device:", device)


## Part 1 — Diagnostic single-seed run

Manual training loop (not `train_one_run`) so we can capture per-epoch loss
directly, rather than the checkpoint object.

In [ ]:
X_train, y_train, X_test, y_test = get_official_split(DATASET)
y_train_enc, y_test_enc, mapping = encode_labels(y_train, y_test)
num_classes = len(mapping)
input_shape = X_train.shape[1:]
print(f"train {X_train.shape}, test {X_test.shape}, num_classes={num_classes}")

set_seed(0)
model = build_model(MODEL_NAME, input_size=input_shape[-1], num_classes=num_classes).to(device)
train_loader, test_loader = build_dataloaders(X_train, y_train_enc, X_test, y_test_enc, batch_size=64)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 50
train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_losses.append(running / len(train_loader))

    model.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            running += criterion(model(xb), yb).item()
    val_losses.append(running / len(test_loader))

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:>3}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val (test set)")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title(f"{MODEL_NAME} on {DATASET} — training curve")
plt.legend()
plt.tight_layout()
plt.show()


If `val_loss` is still dropping at epoch 50, bump `EPOCHS` — the real
multi-seed benchmark run below uses `train_one_run`'s early stopping, so it
won't waste epochs once validation plateaus regardless of what's set here.

## Confusion matrix

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title(f"{MODEL_NAME} on {DATASET} — confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(compute_metrics(y_true, y_pred))


**Known confusion, per the literature review:** row 2 of the literature
table flags walking vs. walking-upstairs as a commonly-confused pair for
plain (non-attention) BiLSTM/BiGRU models. Worth checking whether that same
pair shows up as the largest off-diagonal entries here — if so, that's a
useful thing to callback to when Baseline 2 (attention) results come in,
since attention is specifically supposed to help resolve this kind of
confusion.

## Part 2 — Official multi-seed benchmark run

The real result: 3 seeds, through `train_one_run` (with early stopping),
efficiency profiled once, every run logged to `raw_results.csv`. This is
the exact code path every later model (Baseline 2, CNN-BiGRU, Dual-Attention
BiGRU) goes through, so numbers stay directly comparable.

In [ ]:
SEEDS = [0, 1, 2]
RESULTS_CSV = Path("../results/tables/raw_results.csv")
CHECKPOINT_DIR = Path("../checkpoints") / MODEL_NAME

from src.utils import save_checkpoint

probe = build_model(MODEL_NAME, input_size=input_shape[-1], num_classes=num_classes)
efficiency = profile_model(probe, input_shape, device)
print("efficiency:", efficiency)

run_metrics = []
for seed in SEEDS:
    set_seed(seed)
    model = build_model(MODEL_NAME, input_size=input_shape[-1], num_classes=num_classes)
    train_loader, test_loader = build_dataloaders(X_train, y_train_enc, X_test, y_test_enc, batch_size=64)
    metrics = train_one_run(model, train_loader, test_loader, device, epochs=50, lr=1e-3, patience=8)
    print(f"seed {seed}: {metrics}")
    run_metrics.append(metrics)

    save_checkpoint(model, CHECKPOINT_DIR / f"{DATASET}_official_seed{seed}.pt")
    append_result_row({
        "model": MODEL_NAME, "dataset": DATASET, "split_mode": "official",
        "fold_id": None, "seed": seed, **metrics, **efficiency,
    }, csv_path=RESULTS_CSV)

acc = [m["accuracy"] for m in run_metrics]
f1 = [m["macro_f1"] for m in run_metrics]
print(f"\n{MODEL_NAME} official split -- accuracy: {np.mean(acc):.4f} +/- {np.std(acc):.4f}, "
      f"macro-F1: {np.mean(f1):.4f} +/- {np.std(f1):.4f}")


## Notes / next steps

- [ ] These 3 rows are the first entries in `raw_results.csv` — sanity check
      accuracy is well above chance (1/6 ≈ 0.167) before moving on; if it
      isn't, the issue is more likely in preprocessing than in this model.
- [ ] Checkpoints saved to `checkpoints/bigru_baseline/ucihar_official_seed{0,1,2}.pt`
      (Part 1's diagnostic-run model above is intentionally not saved --
      only the benchmark run's 3 seeds are).
- [ ] `03_baseline_bilstm_attention.ipynb` reuses this notebook's structure
      and explicitly compares Baseline 2 against these numbers.
